In [ ]:
# -*- coding: utf-8 -*-
"""
Use UVDoc (official repo) in-process (no subprocess):
- We execute UVDoc/demo.py within the current interpreter via runpy.
- Input/Output/tmp paths are hard-coded as requested.
- tmp は残します。UVDoc 側が生成する中間/出力も tmp に集約します。

前提:
  - ./UVDoc が存在（https://github.com/tanguymagne/UVDoc）
  - ./UVDoc/model/best_model.pkl が存在（同梱の学習済み）  # README参照
  - 依存は `uv pip install -r ./UVDoc/requirements_demo.txt` 済み
"""
from __future__ import annotations
from pathlib import Path
import os, sys, runpy, shutil, time, traceback
from typing import List, Optional
from PIL import Image, ImageOps

# ====== 0) パス設定（ハードコーディング） ======
INPUT_DIR  = Path("./data/sample1/")   # 処理対象画像をここに置く
OUTPUT_DIR = Path("./output")          # 出力先
TMP_DIR    = Path("./tmp")             # 中間結果（削除しない）
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# UVDoc リポジトリの場所（clone 先）
UVDOC_ROOT = Path("./UVDoc").resolve()
DEMO_PY    = UVDOC_ROOT / "demo.py"
CKPT_PATH  = UVDOC_ROOT / "model" / "best_model.pkl"

import UVDoc
import sys

sys.path.append(str(UVDOC_ROOT))  # UVDoc のパスを追加

# ====== 1) ユーティリティ ======
def _list_images(dirpath: Path) -> List[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    return sorted([p for p in dirpath.glob("*") if p.suffix.lower() in exts])

def _exif_fixed_copy(src: Path, dst: Path) -> None:
    im = Image.open(src)
    im = ImageOps.exif_transpose(im)
    im.save(dst, quality=95)

def _new_files(before: List[Path], after: List[Path]) -> List[Path]:
    bset = {p.resolve() for p in before}
    return [p for p in after if p.resolve() not in bset]

def _pick_best_image(paths: List[Path]) -> Optional[Path]:
    if not paths:
        return None
    # いちばん“それっぽい”もの：サイズ(バイト)と更新時刻の複合で決める
    paths = [p for p in paths if p.suffix.lower() in {".png",".jpg",".jpeg",".tif",".tiff",".webp"}]
    if not paths:
        return None
    def key(p: Path):
        try:
            st = p.stat()
            return (st.st_mtime, st.st_size)
        except FileNotFoundError:
            return (0, 0)
    return max(paths, key=key)

# ====== 2) UVDoc 実行（同一プロセス内） ======
def run_uvdoc_inprocess(img_path: Path, ckpt_path: Path, uvd_root: Path) -> Path:
    """
    UVDoc/demo.py を runpy で実行。
    出力先は demo.py の仕様に従うため、実行前後でディレクトリ差分を取り、新規生成画像を拾う。
    ここでは入出力ともに TMP_DIR を作業ディレクトリにして閉じ込める。
    """
    if not DEMO_PY.exists():
        raise FileNotFoundError(f"UVDoc demo not found: {DEMO_PY}")
    if not ckpt_path.exists():
        raise FileNotFoundError(f"UVDoc checkpoint not found: {ckpt_path}")

    # 作業ディレクトリは tmp（UVDoc の出力を全部ここに落とす）
    prev_cwd = Path.cwd()
    os.chdir(TMP_DIR)
    try:
        # 実行前スナップショット
        before = list(TMP_DIR.rglob("*"))

        # sys.argv を偽装して demo.py を __main__ として走らせる（＝サブプロセスではない）
        argv_backup = sys.argv[:]
        sys.argv = [
            "demo.py",
            "--img-path", str(img_path.resolve()),
            "--ckpt-path", str(ckpt_path.resolve()),
        ]
        # 実行（例外は上位へ）
        runpy.run_path(str(DEMO_PY), run_name="__main__")

        # 実行後スナップショット
        after = list(TMP_DIR.rglob("*"))
    finally:
        # 復旧
        sys.argv = argv_backup
        os.chdir(prev_cwd)

    # 生成物のうち、新規の画像ファイルを拾う
    created = _new_files(before, after)
    out_img = _pick_best_image(created)
    if out_img is None:
        raise RuntimeError("UVDoc demo did not produce an image output under tmp.")
    return out_img

# ====== 3) メインパイプライン ======
def process_one(src_img: Path):
    try:
        stem = src_img.stem
        # 1) EXIF補正して tmp に複製（UVDoc 側は EXIF を見ないため）
        exif_fixed = TMP_DIR / f"{stem}.exif.png"
        _exif_fixed_copy(src_img, exif_fixed)

        # 2) UVDoc 実行（同一プロセス）→ 生成画像のパスを得る
        produced = run_uvdoc_inprocess(exif_fixed, CKPT_PATH, UVDOC_ROOT)

        # 3) 出力へ集約（拡張子はそのまま）
        dst = OUTPUT_DIR / f"{stem}.uvdoc{produced.suffix.lower()}"
        shutil.copy2(produced, dst)
        print(f"[OK] {src_img.name} -> {dst.name}")
    except Exception as e:
        print(f"[ERR] {src_img.name}: {e}")
        traceback.print_exc()

def main():
    imgs = _list_images(INPUT_DIR)
    if not imgs:
        print(f"Place images in: {INPUT_DIR.as_posix()}")
        return
    for p in imgs:
        process_one(p)
    print(f"Done. tmp kept at: {TMP_DIR.as_posix()}")

if __name__ == "__main__":
    main()
